# 06 — Sensitivity (duplicates), uplift segments (E6), experiment design (E7)

**Blocks**

| Block | Question | Contract |
|---|---|---|
| A — Dedup sensitivity | Do conclusions survive exact-row deduplication? (duplicates are non-random: E1 found 4.5% of control vs 17.9% of treated rows involved) | Full data stays primary; dedup is a sensitivity read. ITT recomputed; top-10% policy ordering re-checked on deduped validation rows. |
| B — E6 segments | Where does estimated uplift concentrate, and does the observed randomized effect agree with the ranking? | CATE quantile segments (top 10% / 10-25% / 25-50% / 50-75% / bottom 25%); observed slice effects from randomized arms with bootstrap CIs. |
| C — E7 design + impact | What experiment would validate uplift targeting, and what is the business translation? | Design block + illustrative power sketch (benchmark scale). |

Language discipline: dedup deltas are diagnostics; segment effects are slice-level causal (randomized arms inside slices); E7 impact is benchmark-scale illustration only. All absolute numbers carry the benchmark subsampling caveat (E3/E5).


## Block A — Duplicate sensitivity (story corrected after the collision check)

E1's collision check settled the mechanism: treated rows subsampled to control size duplicate at **4.4%**, statistically identical to the control arm's **4.5%**. The 17.9% treated-arm figure is an **arm-size artifact** — the treated arm is ~5.7× larger, and identical feature vectors (real users on the boundary values of quantile-coded features) collide proportionally to group size. Duplicates are therefore **real users, not duplicate storage**.

Consequence: a naive dedup drops mostly zero-outcome treated rows → the treated rate mechanically rises → the ITT is **inflated** (1.034 → 1.493 pp, ≈ +45% relative). The frozen decision stands: **full data stays primary**; dedup is reported as a sensitivity with the corrected interpretation — *"the naive-dedup effect is the artifact, not the full-data effect."*

In [1]:
import json, time, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
import sys
sys.path.insert(0, r"C:\Users\lucas\Desktop\ds project\Criteo\src")
import config as C
import core
import viz
import matplotlib.pyplot as plt
SEED = 42

df = pd.read_parquet(C.INTERIM / "criteo_uplift_v2_hashed.parquet")
print("full data:", df.shape)


full data: (13979592, 17)


In [2]:
def itt(y):
    y1 = df[y][df[C.TREATMENT] == 1].to_numpy()
    y0 = df[y][df[C.TREATMENT] == 0].to_numpy()
    ate, lo, hi, se = core.ate_bootstrap(y1, y0, n_boot=2000, seed=SEED)
    return dict(rate_t=round(y1.mean() * 100, 4), rate_c=round(y0.mean() * 100, 4),
                itt_pp=round(ate * 100, 4), lo=round(lo * 100, 4), hi=round(hi * 100, 4))

full_itt = {y: itt(y) for y in (C.VISIT, C.CONVERSION)}
content_cols = C.FEATURES + [C.TREATMENT, C.VISIT, C.CONVERSION, C.EXPOSURE]
dd = df.drop_duplicates(subset=content_cols, keep="first")

def itt_dd(yseries, frame):
    y1 = frame[yseries][frame[C.TREATMENT] == 1].to_numpy()
    y0 = frame[yseries][frame[C.TREATMENT] == 0].to_numpy()
    ate, lo, hi, se = core.ate_bootstrap(y1, y0, n_boot=2000, seed=SEED)
    return dict(rate_t=round(y1.mean() * 100, 4), rate_c=round(y0.mean() * 100, 4),
                itt_pp=round(ate * 100, 4), lo=round(lo * 100, 4), hi=round(hi * 100, 4))

dd_itt = {y: itt_dd(y, dd) for y in (C.VISIT, C.CONVERSION)}
rows = []
for y, lab in ((C.VISIT, "visit"), (C.CONVERSION, "conversion")):
    for ver, d in (("full", full_itt[y]), ("dedup", dd_itt[y])):
        rows.append(dict(outcome=lab, version=ver, rate_treated_pp=d["rate_t"],
                         rate_control_pp=d["rate_c"], itt_pp=d["itt_pp"],
                         ci_lo_pp=d["lo"], ci_hi_pp=d["hi"]))
itt_tab = pd.DataFrame(rows)
itt_tab.to_csv(C.TABLES / "e6_dedup_itt_sensitivity.csv", index=False)
print("dedup rows:", len(dd), "| treat ratio dedup:", round(dd[C.TREATMENT].mean(), 5))
itt_tab

dedup rows: 12720047 | treat ratio dedup: 0.839


,outcome,version,rate_treated_pp,rate_control_pp,itt_pp,ci_lo_pp,ci_hi_pp
0,visit,full,4.8543,3.8201,1.0342,1.0060,1.0623
1,visit,dedup,5.4047,3.9116,1.4932,1.4649,1.5228
2,conversion,full,0.3089,0.1938,0.1152,0.1087,0.1221
3,conversion,dedup,0.3440,0.1984,0.1456,0.1389,0.1527


In [3]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.4))
for ax, ycol in zip(axes, ["visit", "conversion"]):
    sub = itt_tab[itt_tab.outcome == ycol]
    full, dsub = sub[sub.version == "full"], sub[sub.version == "dedup"]
    x = np.arange(1); w = 0.34
    ax.bar([-w/2], full.rate_treated_pp, w, color=viz.COLOR_TREATED, label="treated — full")
    ax.bar([+w/2], full.rate_control_pp, w, color=viz.COLOR_CONTROL, label="control — full")
    ax.bar([-w/2], dsub.rate_treated_pp, w, color=viz.COLOR_TREATED, alpha=0.35, hatch="//", label="treated — dedup")
    ax.bar([+w/2], dsub.rate_control_pp, w, color=viz.COLOR_CONTROL, alpha=0.35, hatch="//", label="control — dedup")
    ax.text(0.35, full.rate_treated_pp.iloc[0]*1.05, f"ITT full: {full.itt_pp.iloc[0]:.3f}pp", fontsize=9)
    ax.text(0.35, full.rate_control_pp.iloc[0]*1.35, f"ITT dedup: {dsub.itt_pp.iloc[0]:.3f}pp", fontsize=9, color="dimgray")
    ax.set_xticks([0], ["Visit" if ycol=="visit" else "Conversion"]); ax.set_xlim(-0.65, 0.85)
axes[0].set_title("Naive dedup INFLATES the visit ITT: 1.034 -> 1.493 pp"); axes[0].set_ylabel("rate (%)")
axes[0].legend(fontsize=8); axes[1].set_title("Conversion ITT: 0.115 -> 0.146 pp (same artifact)")
fig.suptitle("Duplicates are feature collisions between real users — naive dedup is the artifact, not the fix", y=1.04, fontweight="bold")
fig.tight_layout()
viz.savefig(fig, "e6_dedup_sensitivity_itt")
print("figure saved")

figure saved


### Deduped validation: policy ordering re-check

In [4]:

# deduped validation rows + E4 CATE scores joined by row_hash
val_h = df.loc[df["row_hash"].mod(1000).between(700, 849), ["row_hash", C.TREATMENT, C.VISIT, C.CONVERSION]].copy()
val_ded = val_h.drop_duplicates(subset=["row_hash"], keep="first")
vs = pd.read_parquet(C.ROOT / "data" / "interim" / "e4_val_uplift_scores.parquet")
vs = vs.drop_duplicates(subset=["row_hash"], keep="first")
mg = val_ded.merge(vs[["row_hash", "cate_s"]], on="row_hash", how="inner")
print("deduped validation:", mg.shape)

deduped validation: (1904659, 5)


In [5]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
tr_mask = (df["row_hash"] % 1000 < 700) & (df[C.TREATMENT] == 1)
vmask = df["row_hash"].isin(val_ded["row_hash"])
m_resp = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, early_stopping=True, random_state=42)
m_resp.fit(df.loc[tr_mask, C.FEATURES].astype("float32"), df.loc[tr_mask, C.VISIT])
vX = df.loc[vmask, C.FEATURES].astype("float32")
resp_pred = m_resp.predict_proba(vX)[:, 1]
vm = df.loc[vmask, ["row_hash"]].copy(); vm["resp_score"] = resp_pred
vm = vm.drop_duplicates("row_hash")
mg = mg.merge(vm, on="row_hash", how="inner")
auc_check = roc_auc_score(df.loc[vmask, C.VISIT], resp_pred)
print("response refit validation AUC:", round(auc_check, 4), "(E2 reference: 0.9466)")
mg.shape

response refit validation AUC: 0.9466 (E2 reference: 0.9466)


(1904659, 6)

In [6]:
pol_rows = []
for pol, col in (("Uplift", "cate_s"), ("Response", "resp_score")):
    thr = mg[col].quantile(0.90)
    sl = mg[mg[col] >= thr]
    y1 = sl[C.VISIT][sl[C.TREATMENT] == 1].to_numpy()
    y0 = sl[C.VISIT][sl[C.TREATMENT] == 0].to_numpy()
    ate, lo, hi, se = core.ate_bootstrap(y1, y0, n_boot=2000, seed=SEED)
    pol_rows.append(dict(policy=pol, budget=0.10, n_sel=len(sl), effect_pp=round(ate * 100, 3),
                         ci_lo_pp=round(lo * 100, 3), ci_hi_pp=round(hi * 100, 3),
                         incremental_per_1000=round(ate * 1000, 1)))
pol_tab = pd.DataFrame(pol_rows)
pol_tab.to_csv(C.TABLES / "e6_dedup_policy_sensitivity.csv", index=False)
pol_tab

,policy,budget,n_sel,effect_pp,ci_lo_pp,ci_hi_pp,incremental_per_1000
0,Uplift,0.1,190466,6.721,6.179,7.301,67.2
1,Response,0.1,190466,4.845,4.243,5.479,48.4


## Block B — E6: uplift segments (validation, full data)

Predicted CATE quantiles vs observed randomized-arm effects inside each segment. Negative estimated CATE does **not** establish individual harm — segment effects are group contrasts only.

In [7]:
bounds = [1.00, 0.90, 0.75, 0.50, 0.25, 0.00]
labels = ["Top 10%", "10-25%", "25-50%", "50-75%", "Bottom 25%"]
seg_rows = []
for i, lab in enumerate(labels):
    hi_cut, lo_cut = mg["cate_s"].quantile(bounds[i]), mg["cate_s"].quantile(bounds[i + 1])
    sl = mg[(mg["cate_s"] >= lo_cut) & (mg["cate_s"] <= hi_cut)]
    y1 = sl[C.VISIT][sl[C.TREATMENT] == 1].to_numpy()
    y0 = sl[C.VISIT][sl[C.TREATMENT] == 0].to_numpy()
    ate, lo, hi, se = core.ate_bootstrap(y1, y0, n_boot=2000, seed=SEED)
    seg_rows.append(dict(segment=lab, pred_cate_pp=round(sl["cate_s"].mean() * 100, 3),
                         obs_effect_pp=round(ate * 100, 3), ci_lo_pp=round(lo * 100, 3),
                         ci_hi_pp=round(hi * 100, 3), n=len(sl), n_treated=len(y1), n_control=len(y0)))
seg_tab = pd.DataFrame(seg_rows)
seg_tab.to_csv(C.TABLES / "e6_uplift_segments.csv", index=False)
seg_tab

,segment,pred_cate_pp,obs_effect_pp,ci_lo_pp,ci_hi_pp,n,n_treated,n_control
0,Top 10%,6.352,6.721,6.179,7.301,190466,165151,25315
1,10-25%,0.713,0.535,0.238,0.844,285699,242431,43268
2,25-50%,0.132,0.125,0.015,0.227,477212,401844,75368
3,50-75%,0.033,0.081,0.039,0.126,476804,400801,76003
4,Bottom 25%,-0.003,0.186,0.127,0.248,476956,389379,87577


In [8]:
fig, ax = plt.subplots(figsize=(8.6, 4.6))
x = np.arange(len(seg_tab)); w = 0.38
ax.bar(x - w/2, seg_tab.pred_cate_pp, w, color="#5d6d7e", label="predicted mean CATE")
ax.bar(x + w/2, seg_tab.obs_effect_pp, w, color=viz.COLOR_UP, label="observed effect (randomized arms)")
ax.errorbar(x + w/2, seg_tab.obs_effect_pp,
            yerr=[seg_tab.obs_effect_pp - seg_tab.ci_lo_pp, seg_tab.ci_hi_pp - seg_tab.obs_effect_pp],
            fmt="none", ecolor="black", capsize=3, label="95% CI")
ax.set_xticks(x, seg_tab.segment)
ax.set_ylabel("effect (pp)")
ax.axhline(1.034, color="red", ls="--", lw=1, label="pooled ITT, full data (+1.03 pp)")
ax.set_title("Heterogeneity is real but concentrated: only the top decile clearly beats the average")
ax.legend(fontsize=8)
fig.tight_layout()
viz.savefig(fig, "e6_uplift_segments")
seg_tab

,segment,pred_cate_pp,obs_effect_pp,ci_lo_pp,ci_hi_pp,n,n_treated,n_control
0,Top 10%,6.352,6.721,6.179,7.301,190466,165151,25315
1,10-25%,0.713,0.535,0.238,0.844,285699,242431,43268
2,25-50%,0.132,0.125,0.015,0.227,477212,401844,75368
3,50-75%,0.033,0.081,0.039,0.126,476804,400801,76003
4,Bottom 25%,-0.003,0.186,0.127,0.248,476956,389379,87577


## Block C — E7: experiment design + business impact (power corrected per review)

**The experiment to run next: uplift targeting vs current targeting, at equal budget.**

| Element | Spec |
|---|---|
| Control | Current targeting (response ranking, top-k) |
| Treatment | Uplift targeting (S-learner CATE, top-k) |
| Constraint | Identical budget & reach (the 10% tier; tight-budget tier 5–10% is where the gap is widest) |
| Design detail | **Each arm carries its own randomized holdout** (the holdout identifies that arm's incremental effect — a difference-in-differences comparison, NOT a two-proportion test) |
| Primary metric | Incremental visits per 1,000 targeted (slice-contrast estimate under randomization) |
| Secondary | Total visits, conversions, cost per incremental conversion |
| Guardrails | Exposure & frequency (measurable), downstream economics (NOT in public data) |
| Success | Pre-registered: uplift > response, one-sided alpha, contrast CI excludes 0 |

E5 evidence feeding this: at 10% budget uplift captures **65.4%** of all incremental visits vs response **47.3%** (+18.2 pp share) and **68.3 vs 56.0** incremental visits per 1,000 targeted on the untouched test split.

**Business impact translation:** on benchmark scale, uplift at 10% budget yields ≈ **+18 more incremental visits per 1,000 targeted** than response targeting (64.0 vs 46.2 on validation). The public data has no revenue/margin/LTV, so no monetary claim is invented: *"In a commercial environment I would replace incremental visits with incremental contribution and optimize the policy against expected profit."*

In [9]:
def n_per_arm_policy(rates, delta, holdout_frac, alpha=0.05, power=0.8):
    """Power for the E7 design: two targeting policies compared at equal budget,
    EACH ARM with its own randomized holdout (fraction `holdout_frac` gets no ad).

    Per arm, the incremental effect is a difference in proportions between the
    targeted share (1-h) and the holdout share (h) of that arm's users. The
    contrast between the two arms' effects is the primary comparison.
    rates: dict policy -> (rate_targeted, rate_holdout) — actual slice rates (E5).
    delta: contrast in incremental effects to detect (proportion units).
    """
    z_a, z_b = 1.959963985, 0.841621234
    s = 1 - holdout_frac
    var_total = 0.0
    for (p_t, p_c) in rates.values():
        var_total += p_t * (1 - p_t) / s + p_c * (1 - p_c) / (1 - s)
    return (z_a + z_b) ** 2 * var_total / delta ** 2

# actual slice arm rates from E5 (validation, 10% budget): read e5_slice_rates.csv
rates_tab = pd.read_csv(r"C:\Users\lucas\Desktop\ds project\Criteo\outputs\tables\e5_slice_rates.csv")
r10 = rates_tab[(rates_tab.outcome == "visit_rates") & (rates_tab.budget == 0.10)].set_index("policy")
rate_uplift = (float(r10.loc["Uplift", "rate_treated_sel"]), float(r10.loc["Uplift", "rate_control_sel"]))
rate_resp = (float(r10.loc["Response", "rate_treated_sel"]), float(r10.loc["Response", "rate_control_sel"]))
delta = abs((rate_uplift[0] - rate_uplift[1]) - (rate_resp[0] - rate_resp[1]))
print(f"slice rates 10% budget - uplift: {rate_uplift[0]:.4f}/{rate_uplift[1]:.4f} | response: {rate_resp[0]:.4f}/{rate_resp[1]:.4f}")
print(f"effect contrast to detect: {delta*100:.2f}pp")

n_50 = n_per_arm_policy({"uplift": rate_uplift, "response": rate_resp}, delta, holdout_frac=0.5)
n_85 = n_per_arm_policy({"uplift": rate_uplift, "response": rate_resp}, delta, holdout_frac=0.15)
# honest reference: broadcast ITT detection (85/15, delta=1.03pp, actual arm rates)
n_bcast = (1.959963985 + 0.841621234) ** 2 * (
    0.048543 * (1 - 0.048543) / 0.85 + 0.038201 * (1 - 0.038201) / 0.15) / 0.010342 ** 2

fig, ax = plt.subplots(figsize=(8, 4.4))
vals = [n_50, n_85, n_bcast]
lbls = ["policy experiment\n50/50 within-arm holdout", "policy experiment\n85/15 within-arm holdout", "broadcast ITT test\n85/15, detect +1.03pp"]
ax.bar(np.arange(3), vals, color=["#1e8449", "#145a32", "#7f8c8d"])
for i_, v in enumerate(vals):
    ax.text(i_, v * 1.03, f"~{int(np.ceil(v/1000))}k users / arm", ha="center", fontweight="bold")
ax.set_xticks(np.arange(3), lbls)
ax.set_ylabel("users per arm (alpha=0.05 two-sided, power=0.80)")
ax.set_title("E7 power (corrected difference-in-differences design):\na policy comparison needs ~41k (50/50) to ~77k (85/15) users per arm, each with its own holdout")
fig.text(0.99, 0.01, "benchmark-scale rates; ignores clustering/multiple testing/noncompliance", ha="right", fontsize=7, color="dimgray")
fig.tight_layout()
viz.savefig(fig, "e7_experiment_power_sketch")
print(f"n/arm: 50/50 holdout ~{n_50:,.0f} | 85/15 holdout ~{n_85:,.0f} | broadcast ITT ~{n_bcast:,.0f}")

slice rates 10% budget - uplift: 0.2743/0.2103 | response: 0.3743/0.3281
effect contrast to detect: 1.78pp


n/arm: 50/50 holdout ~40,797 | 85/15 holdout ~76,801 | broadcast ITT ~21,962


## Block A/B/C verdicts

- **Dedup sensitivity (corrected story):** duplicates are feature-collisions between real users (arm-size artifact, not storage duplication). Naive dedup drops mostly zero-outcome treated rows and **inflates** the visit ITT **1.034 → 1.493 pp** (+45% relative) and conversion 0.115 → 0.146 pp. Full data stays primary — the naive-dedup number is the artifact. Policy ordering is unchanged under dedup (uplift **67.2** vs response **48.5** per 1,000 targeted at 10%), so every comparative conclusion is robust.
- **Slice treatment-share diagnostic:** inside the top-5% / top-10% uplift slices the treated share is **87.7% / 86.7%** (population 83.9% on validation) — a mild tilt consistent with the small detectable imbalance; slice contrasts remain causal under within-test randomization, with the tilt disclosed.
- **Segments:** observed randomized-arm effects confirm the ranking only in the top decile (observed **+6.7 pp** [6.2, 7.3] vs predicted 6.4 pp); mid segments match predictions closely (25–50%: 0.13 pp predicted vs 0.13 pp observed); bottom 25% shows small but **positive** observed effect (+0.19 pp) despite negative mean predicted CATE — negative estimated CATE is NOT evidence of individual harm.
- **E7 (corrected power):** uplift-targeted vs response-targeted at the 10% budget tier is the experiment to run. Power is computed for the ACTUAL design — each policy arm carries its own randomized holdout, using the actual slice arm rates from E5 (uplift 27.4%/21.0%, response 37.4%/32.8%; contrast 1.78 pp): **~41k users/arm with a 50/50 within-arm holdout, ~77k with an 85/15 holdout**. For scale: detecting the broadcast ITT (+1.03 pp) needs ~22k/arm — the policy comparison costs only ~2–3.5× a broadcast test.

### Limitations
- Dedup is a sensitivity read only — the deduped population is not the real-world population (no user id; content-identity based).
- Segment effects are group contrasts; per-user CATE is a proxy (no pointwise guarantees).
- Power calc ignores clustering, multiple testing, and non-compliance; production experiment needs an exposure/frequency guardrail plan.
- All magnitudes benchmark-scale (non-uniform subsampling).
- **Hash note:** the validation-slice re-check dedups by 32-bit `row_hash` (not content): ~20k colliding distinct-row pairs are expected at 14M rows, so a few distinct rows merge — negligible at these effect sizes, and the full-data dedup (content-column exact match) is unaffected.

### Slice treatment-share diagnostic (imbalances where it matters)

In [10]:
# slice treatment-share diagnostic (mild tilt check)
share_rows = []
for b in (0.05, 0.10):
    thr = mg["cate_s"].quantile(1 - b)
    sel = mg[mg["cate_s"] >= thr]
    share_rows.append(dict(slice=f"top {int(b*100)}% uplift", n=len(sel),
                           treated_share=round(sel[C.TREATMENT].mean(), 4),
                           population=round(mg[C.TREATMENT].mean(), 4)))
share_tab = pd.DataFrame(share_rows)
share_tab.to_csv(C.TABLES / "e6_slice_treated_share.csv", index=False)
share_tab

,slice,n,treated_share,population
0,top 5% uplift,95233,0.8774,0.8387
1,top 10% uplift,190466,0.8671,0.8387
